[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/galvezam/eia-data-pipeline/blob/main/processing/natural_gas_crude_processing.ipynb)

# Natural Gas & Crude Oil Processing
**EIA Data Pipeline — Kyle's Processing Notebook**

Datasets:
- `natural_gas_*.csv` — Monthly marketed production by state (MMcf)
- `natural_gas_trade_*.csv` — Annual state-level exports, imports, and interstate movements (MMcf)
- `crude_oil_imports_*.csv` — Monthly crude oil imports by country of origin → refinery state (thousand barrels)

Pipeline steps: **Clean → Transform → Analyze → Upload Parquet to S3**

## 0. Environment Setup & Configuration

In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

# Install Spark with Hadoop as the underlying scheduler
!wget -q https://downloads.apache.org/spark/spark-3.5.8/spark-3.5.8-bin-hadoop3.tgz
!tar xf spark-3.5.8-bin-hadoop3.tgz
!ls -l

os.environ["SPARK_HOME"] = "spark-3.5.8-bin-hadoop3"
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages org.apache.hadoop:hadoop-aws:3.3.4,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.262 "
    "pyspark-shell"
)

!pip install -q findspark
import findspark
findspark.init()
print("Spark environment ready.")

In [ ]:
# ── Upload your 3 CSV files from your local machine ───────────────────────
from google.colab import files

print("Upload: natural_gas_*.csv, natural_gas_trade_*.csv, crude_oil_imports_*.csv")
uploaded = files.upload()   # a file picker will pop up — select all 3 at once
print("Uploaded:", list(uploaded.keys()))

In [ ]:
import sys

# AWS credentials — fill in here if using Section 5 (S3 upload)
AWS_ACCESS_KEY = ""
AWS_SECRET_KEY = ""
AWS_REGION     = ""
BUCKET_NAME    = ""

# Uploaded files land in /content/ in Colab
NAT_GAS_PATH   = next(f for f in uploaded if "natural_gas" in f and "trade" not in f)
NG_TRADE_PATH  = next(f for f in uploaded if "natural_gas_trade" in f)
CRUDE_PATH     = next(f for f in uploaded if "crude_oil" in f)

# ── S3 output prefixes ─────────────────────────────────────────────────────
S3_BASE = f"s3a://{BUCKET_NAME}/processed"

S3_NG_PRODUCTION   = f"{S3_BASE}/natural_gas_production/"
S3_NG_INTL_TRADE   = f"{S3_BASE}/natural_gas_trade_international/"
S3_NG_INTERSTATE   = f"{S3_BASE}/natural_gas_trade_interstate/"
S3_CRUDE_BY_ORIGIN = f"{S3_BASE}/crude_oil_imports_by_state_country/"
S3_CRUDE_STATE     = f"{S3_BASE}/crude_oil_imports_by_state/"
S3_CRUDE_GRADE     = f"{S3_BASE}/crude_oil_imports_grade_breakdown/"

print("Config loaded.")
print(f"  Natural gas:   {NAT_GAS_PATH}")
print(f"  NG trade:      {NG_TRADE_PATH}")
print(f"  Crude imports: {CRUDE_PATH}")
print(f"  S3 bucket:     {BUCKET_NAME}")

## 1. Initialize Spark Session

In [ ]:
from pyspark.sql import SparkSession

import os
print(os.environ.get('JAVA_HOME'))  # Should show the jdk-17 path


builder = (
    SparkSession.builder
    .appName('EIA Natural Gas & Crude Oil Processing')
    .config('spark.hadoop.validateOutputSpecs', False)
)

# Only wire up S3 if credentials are filled in — lets you run locally without AWS
if AWS_ACCESS_KEY and AWS_SECRET_KEY and AWS_REGION:
    builder = (
        builder
        .config('spark.hadoop.fs.s3a.impl',        'org.apache.hadoop.fs.s3a.S3AFileSystem')
        .config('spark.hadoop.fs.s3a.endpoint',    f's3.{AWS_REGION}.amazonaws.com')
        .config('spark.hadoop.fs.s3a.access.key',  AWS_ACCESS_KEY)
        .config('spark.hadoop.fs.s3a.secret.key',  AWS_SECRET_KEY)
    )
    print('S3 configured.')
else:
    print('No S3 credentials — local mode only (Sections 0–4).')

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} ready.')

## 2. Load Raw Data

In [ ]:
def read_csv(path):
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("multiLine", True)   # handles quoted commas in city names
        .option("escape", '"')
        .csv(path)
    )

raw_ng    = read_csv(NAT_GAS_PATH)
raw_trade = read_csv(NG_TRADE_PATH)
raw_crude = read_csv(CRUDE_PATH)

print(f"Natural Gas Production raw rows : {raw_ng.count():>8,}")
print(f"Natural Gas Trade raw rows       : {raw_trade.count():>8,}")
print(f"Crude Oil Imports raw rows       : {raw_crude.count():>8,}")

In [ ]:
print("=== Natural Gas Production Schema ===")
raw_ng.printSchema()

print("\n=== Natural Gas Trade Schema ===")
raw_trade.printSchema()

print("\n=== Crude Oil Imports Schema ===")
raw_crude.printSchema()

## 3. Cleaning

### 3a. Natural Gas Production

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

ng_clean = (
    raw_ng
    # ── Remove rows with no reported production value ──────────────────────
    .filter(F.col("value").isNotNull())
    # ── Keep state-level rows only (duoarea starts with "S") ───────────────
    # Drops U.S. total (NUS), federal offshore (R3FM), and non-state entries
    .filter(F.col("duoarea").startswith("S"))
    # ── Keep only Marketed Production series (the primary production metric)─
    .filter(F.col("process") == "VGM")
    # ── Parse period ("YYYY-MM") to a proper date (first of month) ─────────
    .withColumn("period", F.to_date(F.col("period"), "yyyy-MM"))
    # ── Cast value to double ───────────────────────────────────────────────
    .withColumn("production_mmcf", F.col("value").cast("double"))
    # ── Rename and select relevant columns ─────────────────────────────────
    .withColumnRenamed("area-name", "state_name")
    .select("period", "duoarea", "state_name", "production_mmcf")
    # ── Drop duplicates that may arise from append-style file generation ───
    .dropDuplicates()
)

print(f"Natural Gas Production cleaned rows: {ng_clean.count():,}")
ng_clean.show(5)

### 3b. Natural Gas Trade

In [ ]:
# Map process codes to readable labels
PROCESS_LABELS = {
    "EEI": "Exports (Intl)",
    "IMI": "Imports (Intl)",
    "IMB": "Net Intl Movements",
    "MID": "Interstate Deliveries",
    "MIN": "Net Interstate Receipts",
    "MIE": "Net Intl + Interstate",
}
process_map = F.create_map(
    *[item for pair in [(F.lit(k), F.lit(v)) for k, v in PROCESS_LABELS.items()] for item in pair]
)

trade_clean = (
    raw_trade
    # ── Remove rows with null value ────────────────────────────────────────
    .filter(F.col("value").isNotNull())
    # ── Keep state-level duoarea rows only ─────────────────────────────────
    .filter(F.col("duoarea").rlike(r"^S[A-Z]{2}"))
    # ── Keep only the 6 process codes of interest ──────────────────────────
    .filter(F.col("process").isin(list(PROCESS_LABELS.keys())))
    # ── Cast period to integer (annual data) ───────────────────────────────
    .withColumn("year", F.col("period").cast(IntegerType()))
    # ── Add human-readable process label ──────────────────────────────────
    .withColumn("process_label", process_map[F.col("process")])
    # ── Rename and select ──────────────────────────────────────────────────
    .withColumnRenamed("area-name", "state_name")
    .withColumn("value_mmcf", F.col("value").cast("double"))
    .select("year", "duoarea", "state_name", "process", "process_label", "value_mmcf", "units")
    .dropDuplicates()
)

print(f"Natural Gas Trade cleaned rows: {trade_clean.count():,}")
trade_clean.show(5)

### 3c. Crude Oil Imports

In [ ]:
crude_clean = (
    raw_crude
    # ── Remove rows with null quantity ─────────────────────────────────────
    .filter(F.col("quantity").isNotNull())
    # ── Filter to Refinery State level only to avoid double-counting ───────
    # The raw data repeats each import at 7 geographic levels:
    # Port PADD, Port State, Port, Refinery, Refinery PADD, Refinery State, US total.
    # We keep "Refinery State" as the most useful state-level grain.
    .filter(F.col("destinationTypeName") == "Refinery State")
    # ── Keep only actual country origins (not OPEC/Non-OPEC aggregates) ────
    .filter(F.col("originType") == "CTY")
    # ── Parse period to date ───────────────────────────────────────────────
    .withColumn("period", F.to_date(F.col("period"), "yyyy-MM"))
    # ── Cast quantity ──────────────────────────────────────────────────────
    .withColumn("quantity_thousand_bbl", F.col("quantity").cast("double"))
    # ── Rename destination column for clarity ──────────────────────────────
    .withColumnRenamed("destinationName", "refinery_state")
    .withColumnRenamed("originName",       "origin_country")
    .withColumnRenamed("gradeName",        "crude_grade")
    .select(
        "period", "origin_country", "originId",
        "refinery_state", "destinationId",
        "crude_grade", "quantity_thousand_bbl"
    )
    .dropDuplicates()
)

print(f"Crude Oil Imports cleaned rows: {crude_clean.count():,}")
crude_clean.show(5)

## 4. Analytics

### 4a. Natural Gas: State Production Over Time & % of US Total

In [ ]:
# ── US monthly total (from cleaned state-level data) ──────────────────────
us_ng_total = (
    ng_clean
    .groupBy("period")
    .agg(F.sum("production_mmcf").alias("us_total_mmcf"))
)

# ── Join state totals with US total, compute % share ──────────────────────
ng_state_production = (
    ng_clean
    .join(us_ng_total, on="period", how="left")
    .withColumn(
        "pct_us_production",
        F.round(F.col("production_mmcf") / F.col("us_total_mmcf") * 100, 4)
    )
    .select(
        "period", "duoarea", "state_name",
        "production_mmcf", "us_total_mmcf", "pct_us_production"
    )
    .orderBy("period", F.col("production_mmcf").desc())
)

print(f"State production rows: {ng_state_production.count():,}")
ng_state_production.show(10)

### 4b. Natural Gas: International Trade Summary by State (Annual)

In [ ]:
intl_processes = ["EEI", "IMI", "IMB"]

ng_intl_trade = (
    trade_clean
    .filter(F.col("process").isin(intl_processes))
    .groupBy("year", "duoarea", "state_name", "process", "process_label")
    .agg(F.sum("value_mmcf").alias("total_mmcf"))
    .orderBy("year", "state_name", "process")
)

# ── Pivot wider: one row per state-year, columns for each process ──────────
ng_intl_trade_wide = (
    ng_intl_trade
    .groupBy("year", "duoarea", "state_name")
    .pivot("process", intl_processes)
    .agg(F.first("total_mmcf"))
    .withColumnRenamed("EEI", "exports_mmcf")
    .withColumnRenamed("IMI", "imports_mmcf")
    .withColumnRenamed("IMB", "net_intl_mmcf")
    .orderBy("year", "state_name")
)

print(f"International trade rows (wide): {ng_intl_trade_wide.count():,}")
ng_intl_trade_wide.show(10)

### 4c. Natural Gas: Interstate Movements by State (Annual)

In [ ]:
interstate_processes = ["MID", "MIN"]

ng_interstate = (
    trade_clean
    .filter(F.col("process").isin(interstate_processes))
    .groupBy("year", "duoarea", "state_name", "process", "process_label")
    .agg(F.sum("value_mmcf").alias("total_mmcf"))
    .orderBy("year", "state_name", "process")
)

ng_interstate_wide = (
    ng_interstate
    .groupBy("year", "duoarea", "state_name")
    .pivot("process", interstate_processes)
    .agg(F.first("total_mmcf"))
    .withColumnRenamed("MID", "interstate_delivered_mmcf")
    .withColumnRenamed("MIN", "net_interstate_received_mmcf")
    .orderBy("year", "state_name")
)

print(f"Interstate movements rows (wide): {ng_interstate_wide.count():,}")
ng_interstate_wide.show(10)

### 4d. Crude Oil: State Import Volume by Country of Origin (Monthly)

In [ ]:
crude_by_origin = (
    crude_clean
    .groupBy("period", "refinery_state", "origin_country")
    .agg(F.sum("quantity_thousand_bbl").alias("total_thousand_bbl"))
    .orderBy("period", "refinery_state", F.col("total_thousand_bbl").desc())
)

print(f"Crude imports by state & origin rows: {crude_by_origin.count():,}")
crude_by_origin.show(10)

### 4e. Crude Oil: Total State Imports Over Time & % of US Total

In [ ]:
# ── US monthly total ───────────────────────────────────────────────────────
us_crude_total = (
    crude_clean
    .groupBy("period")
    .agg(F.sum("quantity_thousand_bbl").alias("us_total_thousand_bbl"))
)

# ── State totals + % of US ────────────────────────────────────────────────
crude_by_state = (
    crude_clean
    .groupBy("period", "refinery_state")
    .agg(F.sum("quantity_thousand_bbl").alias("total_thousand_bbl"))
    .join(us_crude_total, on="period", how="left")
    .withColumn(
        "pct_us_imports",
        F.round(F.col("total_thousand_bbl") / F.col("us_total_thousand_bbl") * 100, 4)
    )
    .orderBy("period", F.col("total_thousand_bbl").desc())
)

print(f"Crude imports by state rows: {crude_by_state.count():,}")
crude_by_state.show(10)

### 4f. Crude Oil: Grade Breakdown per State (Monthly)

In [ ]:
# ── State-month totals (for computing within-state grade share) ────────────
state_month_totals = (
    crude_clean
    .groupBy("period", "refinery_state")
    .agg(F.sum("quantity_thousand_bbl").alias("state_total_thousand_bbl"))
)

crude_grade_breakdown = (
    crude_clean
    .groupBy("period", "refinery_state", "crude_grade")
    .agg(F.sum("quantity_thousand_bbl").alias("grade_quantity_thousand_bbl"))
    .join(state_month_totals, on=["period", "refinery_state"], how="left")
    .withColumn(
        "pct_of_state_imports",
        F.round(F.col("grade_quantity_thousand_bbl") / F.col("state_total_thousand_bbl") * 100, 4)
    )
    .orderBy("period", "refinery_state", F.col("grade_quantity_thousand_bbl").desc())
)

print(f"Grade breakdown rows: {crude_grade_breakdown.count():,}")
crude_grade_breakdown.show(10)

## 5. Write Parquet to S3

**Requires a configured S3 bucket.** Fill in `example_config.py` first.  
All cells above this point run entirely locally without AWS credentials.

In [ ]:
def write_parquet(df, path, label):
    """Write a DataFrame to S3 as snappy-compressed Parquet."""
    print(f"Writing {label} → {path} ...")
    df.write.mode("overwrite").option("compression", "snappy").parquet(path)
    print(f"  ✓ Done")

write_parquet(ng_state_production,   S3_NG_PRODUCTION,   "Natural Gas State Production")
write_parquet(ng_intl_trade_wide,    S3_NG_INTL_TRADE,   "Natural Gas International Trade")
write_parquet(ng_interstate_wide,    S3_NG_INTERSTATE,   "Natural Gas Interstate Movements")
write_parquet(crude_by_origin,       S3_CRUDE_BY_ORIGIN, "Crude Imports by State & Country")
write_parquet(crude_by_state,        S3_CRUDE_STATE,     "Crude Imports by State")
write_parquet(crude_grade_breakdown, S3_CRUDE_GRADE,     "Crude Imports Grade Breakdown")

print("\nAll datasets written to S3 successfully.")

## 6. Stop Spark

In [ ]:
spark.stop()
print("Spark session stopped.")